In [ ]:
import pandas as pd

oferta = pd.read_csv("oferta_historica (2).csv", index_col=0, parse_dates=["fecha"])
demanda = pd.read_csv("demanda_historica(in).csv", parse_dates=["fecha"])
hospitales = pd.read_csv("hospitales (1).csv")  
flota = pd.read_csv("flota (1).csv", index_col=0)
parametros = pd.read_csv("parametros(in).csv", index_col="parametro")["valor"]

oferta = oferta.sort_values("fecha").reset_index(drop=True)

print("Oferta:", oferta.shape, oferta["fecha"].min(), "->", oferta["fecha"].max())
print("Nulos:", oferta.isnull().sum().sum())
print("Duplicados fecha:", oferta["fecha"].duplicated().sum())
print("Oferta negativa:", (oferta["oferta"] < 0).sum())
print()
print(oferta["oferta"].describe(percentiles=[0.05,0.25,0.5,0.75,0.95]))

In [ ]:

demanda_total = demanda.groupby("fecha")["demanda"].sum().reset_index()
demanda_total.columns = ["fecha", "demanda_total"]

df = pd.merge(oferta, demanda_total, on="fecha", how="inner")
df["gap"] = df["oferta"] - df["demanda_total"]

print(df.shape)
print(df[["oferta", "demanda_total", "gap"]].describe())
print()
print("Días con gap >= 0 (superávit):", (df["gap"] >= 0).sum(), "de", len(df))
print("% días con déficit:", (df["gap"] < 0).mean() * 100)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["fecha"], df["demanda_total"], label="Demanda total", color="tab:red", linewidth=0.8)
ax.plot(df["fecha"], df["oferta"], label="Oferta", color="tab:blue", linewidth=0.8)
ax.set_title("Oferta vs Demanda total de la red (datos corregidos)")
ax.set_xlabel("Fecha")
ax.set_ylabel("Unidades")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
df["mes"] = df["fecha"].dt.month
df["dia_semana"] = df["fecha"].dt.day_name()
df["anio"] = df["fecha"].dt.year

oferta_por_mes = df.groupby("mes")["oferta"].mean()
oferta_por_dia_semana = df.groupby("dia_semana")["oferta"].mean().reindex(
    ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
)
oferta_por_anio = df.groupby("anio")["oferta"].mean()

print("Por mes:")
print(oferta_por_mes)
print()
print("Por día de semana:")
print(oferta_por_dia_semana)
print()
print("Por año:")
print(oferta_por_anio)

In [ ]:
for lag in [1, 2, 3, 7, 14, 30]:
    autocorr = df["oferta"].autocorr(lag=lag)
    print(f"Rezago {lag} día(s): autocorrelación = {autocorr:.3f}")

In [ ]:
correlacion = df["oferta"].corr(df["demanda_total"])
print(f"Correlación oferta-demanda total: {correlacion:.3f}")

In [ ]:
import numpy as np
from scipy import stats

datos = df["oferta"].values

media_oferta = df["oferta"].mean()
var_oferta = df["oferta"].var()
print(f"Media: {media_oferta:.2f}")
print(f"Varianza: {var_oferta:.2f}")
print(f"Varianza/Media: {var_oferta/media_oferta:.2f}")

normal_params = stats.norm.fit(datos)
gamma_params = stats.gamma.fit(datos, floc=0)

ll_normal = np.sum(stats.norm.logpdf(datos, *normal_params))
ll_gamma = np.sum(stats.gamma.logpdf(datos, *gamma_params))

aic_normal = 2*2 - 2*ll_normal
aic_gamma = 2*3 - 2*ll_gamma

print(f"\nNormal: params={normal_params}, AIC={aic_normal:.2f}")
print(f"Gamma: params={gamma_params}, AIC={aic_gamma:.2f}")

p = media_oferta / var_oferta
n = media_oferta**2 / (var_oferta - media_oferta)
ll_nbinom = np.sum(stats.nbinom.logpmf(datos.astype(int), n, p))
aic_nbinom = 2*2 - 2*ll_nbinom
print(f"Binomial Negativa: n={n:.3f}, p={p:.3f}, AIC={aic_nbinom:.2f}")

In [ ]:
p10_oferta = df["oferta"].quantile(0.10)
df["oferta_baja"] = (df["oferta"] < p10_oferta).astype(int)

prob_base = df["oferta_baja"].mean()
df["oferta_baja_ayer"] = df["oferta_baja"].shift(1)
prob_condicional = df.loc[df["oferta_baja_ayer"] == 1, "oferta_baja"].mean()

print(f"P10 oferta: {p10_oferta}")
print(f"Probabilidad base de día bajo: {prob_base:.3f}")
print(f"Probabilidad de día bajo | ayer fue bajo: {prob_condicional:.3f}")
print(f"Lift: {prob_condicional/prob_base:.3f}")

In [ ]:
df["racha_id"] = (df["oferta_baja"] != df["oferta_baja"].shift(1)).cumsum()
rachas = df[df["oferta_baja"] == 1].groupby("racha_id").size()

print("\nRacha máxima de oferta baja:", rachas.max(), "días")
print("Duración promedio de racha:", rachas.mean().round(2), "días")
print("Número de rachas:", len(rachas))
print("\nDistribución de duraciones:")
print(rachas.value_counts().sort_index())

In [ ]:

df["semana"] = df["fecha"].dt.to_period("W")
df["mes_periodo"] = df["fecha"].dt.to_period("M")

gap_semanal = df.groupby("semana")["gap"].sum()
gap_mensual = df.groupby("mes_periodo")["gap"].sum()

print("=== Gap semanal ===")
print(f"Semanas con déficit neto (gap<0): {(gap_semanal < 0).sum()} de {len(gap_semanal)}")
print(f"% semanas con déficit neto: {(gap_semanal < 0).mean()*100:.1f}%")
print(gap_semanal.describe())

print("\n=== Gap mensual ===")
print(f"Meses con déficit neto (gap<0): {(gap_mensual < 0).sum()} de {len(gap_mensual)}")
print(f"% meses con déficit neto: {(gap_mensual < 0).mean()*100:.1f}%")
print(gap_mensual.describe())

In [ ]:

deficit_dias = df[df["gap"] < 0]["gap"]

print(f"Días con déficit: {len(deficit_dias)}")
print(deficit_dias.describe())

import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.hist(deficit_dias, bins=25, edgecolor="black", color="tab:red")
plt.title("Distribución del déficit (días con gap < 0)")
plt.xlabel("Magnitud del déficit (unidades)")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(df["oferta"], bins=30, edgecolor="black", color="tab:blue")
plt.title("Distribución de la oferta diaria")
plt.xlabel("Oferta diaria (unidades)")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

meses_nombre = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",
                 7:"Jul",8:"Ago",9:"Sep",10:"Oct",11:"Nov",12:"Dic"}

plt.figure(figsize=(8, 5))
plt.plot(oferta_por_mes.index, oferta_por_mes.values, marker="o")
plt.xticks(list(meses_nombre.keys()), list(meses_nombre.values()))
plt.title("Oferta promedio por mes")
plt.xlabel("Mes")
plt.ylabel("Oferta promedio diaria")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df["anio_mes"] = df["fecha"].dt.to_period("M")
oferta_mensual_serie = df.groupby("anio_mes")["oferta"].mean()

plt.figure(figsize=(14, 5))
plt.plot(oferta_mensual_serie.index.astype(str), oferta_mensual_serie.values, marker="o")
plt.xticks(rotation=90)
plt.title("Oferta promedio mensual (2022-2024)")
plt.xlabel("Mes")
plt.ylabel("Oferta promedio diaria")
plt.tight_layout()
plt.show()

In [ ]:
from pandas.plotting import autocorrelation_plot
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
autocorrelation_plot(df["oferta"])
plt.title("Autocorrelación de la oferta diaria")
plt.xlim(0, 60)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

lags = list(range(1, 91))
valores = [df["oferta"].autocorr(lag=l) for l in lags]

plt.figure(figsize=(10, 5))
plt.bar(lags, valores, color="tab:blue", width=0.7)
plt.axhline(0.48, color="red", linestyle="--", linewidth=1, label="0,48 (rezago 1 día)")
plt.axhline(0.45, color="orange", linestyle="--", linewidth=1, label="0,45 (referencia 2 semanas)")
plt.axvline(14, color="gray", linestyle=":", linewidth=1, label="14 días (2 semanas)")
plt.title("Autocorrelación de la oferta diaria por rezago")
plt.xlabel("Rezago (días)")
plt.ylabel("Autocorrelación")
plt.ylim(0, 0.6)
plt.legend()
plt.tight_layout()
plt.show()